# Penggunaan RNN Tingkat Lanjut: Peramalan Suhu Jena

**Buku:** *Deep Learning dengan Python* — Bab 6, Deep Learning untuk Teks dan Sekuens

Memprediksi suhu 24 jam ke depan. Kita membandingkan baseline akal sehat, jaringan Dense, GRU, dropout rekuren, GRU bertumpuk, dan RNN dua arah.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- Generator Python buatan sendiri + `fit_generator` → `keras.utils.timeseries_dataset_from_array` + `fit`.
- Target dibiarkan dalam °C sehingga MAE langsung terbaca dalam derajat (asli: harus dikali simpangan baku).
- Jendela masukan 5 hari (120 titik per jam) agar lebih cepat; asli 10 hari.
- Loss `mse` + metrik `mae`; model terbaik diuji pada data **uji** di akhir.
- Catatan kinerja: `recurrent_dropout > 0` menonaktifkan kernel cuDNN sehingga pelatihan jauh lebih lambat di GPU.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

In [ ]:
import zipfile, numpy as np

URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
jalur_zip = keras.utils.get_file("jena_climate_2009_2016.csv.zip", origin=URL)
with zipfile.ZipFile(jalur_zip) as zf:
    teks = zf.read("jena_climate_2009_2016.csv").decode("utf-8")

baris = teks.strip().split("\n")
header = [h.strip('"') for h in baris[0].split(",")]
baris = baris[1:]
data_mentah = np.array([[float(v) for v in b.split(",")[1:]] for b in baris], dtype="float32")
suhu = data_mentah[:, 1].copy()                     # kolom "T (degC)"
print(len(baris), "baris |", header[1:4], "...")

N_LATIH, N_VAL = 200_000, 100_000                    # sama dengan pembagian asli
rerata = data_mentah[:N_LATIH].mean(axis=0)
simpangan = data_mentah[:N_LATIH].std(axis=0)
data = (data_mentah - rerata) / simpangan            # masukan dinormalisasi; target tetap °C

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3)); plt.plot(suhu); plt.title("Suhu seluruh periode (°C)"); plt.show()
plt.figure(figsize=(10, 3)); plt.plot(suhu[:1440]); plt.title("10 hari pertama (°C)"); plt.show()

In [ ]:
def buat_dataset(laju_sampel=6, panjang=120, batch=256):
    """Masukan: `panjang` titik dengan jarak `laju_sampel` langkah (10 menit/langkah).
    Target: suhu 24 jam setelah titik terakhir."""
    tunda = laju_sampel * (panjang - 1) + 144        # 144 langkah = 24 jam
    umum = dict(sampling_rate=laju_sampel, sequence_length=panjang, batch_size=batch)
    latih = keras.utils.timeseries_dataset_from_array(
        data[:-tunda], targets=suhu[tunda:], shuffle=True,
        start_index=0, end_index=N_LATIH, **umum)
    val = keras.utils.timeseries_dataset_from_array(
        data[:-tunda], targets=suhu[tunda:],
        start_index=N_LATIH, end_index=N_LATIH + N_VAL, **umum)
    uji = keras.utils.timeseries_dataset_from_array(
        data[:-tunda], targets=suhu[tunda:], start_index=N_LATIH + N_VAL, **umum)
    return latih, val, uji

ds_latih, ds_val, ds_uji = buat_dataset()
for sampel, target in ds_latih.take(1):
    print("Sampel:", sampel.shape, "| target:", target.shape)

## 1. Baseline akal sehat
"Suhu 24 jam lagi sama dengan suhu sekarang."

In [ ]:
def mae_baseline(dataset):
    total_galat, n = 0.0, 0
    for sampel, target in dataset:
        sampel = keras.ops.convert_to_numpy(sampel)
        prediksi = sampel[:, -1, 1] * simpangan[1] + rerata[1]
        total_galat += np.abs(prediksi - keras.ops.convert_to_numpy(target)).sum()
        n += len(target)
    return total_galat / n

print(f"MAE baseline validasi: {mae_baseline(ds_val):.2f} °C | uji: {mae_baseline(ds_uji):.2f} °C")

In [ ]:
from keras import layers

def latih(model, epochs=10, ds_l=None, ds_v=None, nama="model"):
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    cb = [keras.callbacks.ModelCheckpoint(f"jena_{nama}.keras", save_best_only=True)]
    ds_l = ds_latih if ds_l is None else ds_l
    ds_v = ds_val if ds_v is None else ds_v
    r = model.fit(ds_l, epochs=epochs, validation_data=ds_v, callbacks=cb)
    h = r.history
    plt.plot(range(1, len(h["mae"]) + 1), h["mae"], "o", ms=3, label="MAE latih")
    plt.plot(range(1, len(h["mae"]) + 1), h["val_mae"], "-", label="MAE validasi")
    plt.axhline(mae_baseline(ds_val), color="gray", ls="--", label="baseline")
    plt.xlabel("Epoch"); plt.ylabel("MAE (°C)"); plt.title(nama); plt.legend(); plt.show()
    print(f"[{nama}] MAE validasi terbaik: {min(h['val_mae']):.2f} °C")
    return model

BENTUK = (120, data.shape[-1])

## 2. Model Dense kecil (urutan waktu diratakan)

In [ ]:
latih(keras.Sequential([keras.Input(shape=BENTUK), layers.Flatten(),
                        layers.Dense(16, activation="relu"), layers.Dense(1)]), nama="dense")

## 3. Baseline rekuren: GRU

In [ ]:
latih(keras.Sequential([keras.Input(shape=BENTUK), layers.GRU(32), layers.Dense(1)]), nama="gru")

## 4. Dropout rekuren
Mask dropout yang **sama** di setiap langkah waktu (Gal, 2015).

In [ ]:
latih(keras.Sequential([keras.Input(shape=BENTUK),
                        layers.GRU(32, recurrent_dropout=0.25),
                        layers.Dropout(0.5), layers.Dense(1)]),
      epochs=20, nama="gru_dropout")

## 5. GRU bertumpuk

In [ ]:
latih(keras.Sequential([keras.Input(shape=BENTUK),
                        layers.GRU(32, recurrent_dropout=0.5, return_sequences=True),
                        layers.GRU(32, recurrent_dropout=0.5),
                        layers.Dropout(0.5), layers.Dense(1)]),
      epochs=20, nama="gru_bertumpuk")

## 6. Urutan terbalik dan RNN dua arah
Untuk cuaca, masa lalu terdekat paling informatif, sehingga urutan terbalik merugikan.

In [ ]:
balik = lambda x, y: (x[:, ::-1, :], y)
latih(keras.Sequential([keras.Input(shape=BENTUK), layers.GRU(32), layers.Dense(1)]),
      ds_l=ds_latih.map(balik), ds_v=ds_val.map(balik), nama="gru_terbalik")

latih(keras.Sequential([keras.Input(shape=BENTUK),
                        layers.Bidirectional(layers.GRU(16)), layers.Dense(1)]),
      nama="gru_dua_arah")

## 7. Evaluasi akhir pada data uji

In [ ]:
for nama in ["dense", "gru", "gru_dropout", "gru_bertumpuk", "gru_dua_arah"]:
    m = keras.models.load_model(f"jena_{nama}.keras")
    print(f"{nama:15s} MAE uji: {m.evaluate(ds_uji, verbose=0)[1]:.2f} °C")

## 8. Bonus: teks juga bisa dibaca terbalik
Pada sentimen IMDB, LSTM dengan urutan terbalik berkinerja hampir sama, dan LSTM dua arah sedikit lebih baik.

In [ ]:
from keras.datasets import imdb

MAKS_KATA = 10000     # kosakata: 10.000 kata paling sering
PANJANG = 500         # potong ulasan setelah 500 kata
(x_latih, y_latih), (x_uji, y_uji) = imdb.load_data(num_words=MAKS_KATA)
x_latih = keras.utils.pad_sequences(x_latih, maxlen=PANJANG)
x_uji = keras.utils.pad_sequences(x_uji, maxlen=PANJANG)
print(x_latih.shape, x_uji.shape)

In [ ]:
def lstm_imdb(dua_arah=False):
    rekuren = layers.Bidirectional(layers.LSTM(32)) if dua_arah else layers.LSTM(32)
    m = keras.Sequential([keras.Input(shape=(None,), dtype="int64"),
                          layers.Embedding(MAKS_KATA, 32), rekuren,
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
    return m

r_balik = lstm_imdb().fit(x_latih[:, ::-1], y_latih, epochs=10, batch_size=128, validation_split=0.2, verbose=0)
r_dua = lstm_imdb(True).fit(x_latih, y_latih, epochs=10, batch_size=128, validation_split=0.2, verbose=0)
print(f"LSTM terbalik  : val acc terbaik {max(r_balik.history['val_accuracy']):.2%}")
print(f"LSTM dua arah  : val acc terbaik {max(r_dua.history['val_accuracy']):.2%}")